<a href="https://colab.research.google.com/github/christinChris/gaiascope/blob/main/notebooks/BurnScope_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install -q solara

import ee, geemap
from google.colab import auth, output

auth.authenticate_user()
ee.Initialize(project="ee-christilekka")
output.enable_custom_widget_manager()
print("Connected")

Connected


In [39]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Land cover ----------
WORLDCOVER = ee.ImageCollection("ESA/WorldCover/v200").first()

classes = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
           50: "Built-up", 60: "Bare / sparse", 70: "Snow and ice",
           80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves",
           100: "Moss and lichen"}

LC_COLORS = {10: "#006400", 20: "#FFBB22", 30: "#FFFF4C", 40: "#F096FF",
             50: "#FA0000", 60: "#B4B4B4", 70: "#F0F0F0", 80: "#0064C8",
             90: "#0096A0", 95: "#00CF75", 100: "#FAE6A0"}

NAME_COLORS = {classes[k]: v for k, v in LC_COLORS.items()}

# ---------- Burn severity (USGS dNBR classes) ----------
SEVERITY   = {1: "Low", 2: "Moderate-low", 3: "Moderate-high", 4: "High"}
SEV_COLORS = ["#FFD54F", "#FF8F00", "#D84315", "#7F0000"]


# ---------- Tables and charts ----------
def color_cell(name):
    c = NAME_COLORS.get(name)
    return f"border-left: 14px solid {c}; padding-left: 8px" if c else ""


def lc_table(groups):
    rows = [{"Land cover": classes[g["class"]], "Area (km²)": g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows)
    df = df[df["Area (km²)"] >= 0.05]
    df = df.sort_values("Area (km²)", ascending=False).reset_index(drop=True)
    df["Share (%)"] = df["Area (km²)"] / df["Area (km²)"].sum() * 100
    df.loc[len(df)] = ["Total", df["Area (km²)"].sum(), 100.0]
    return (df.style.format(precision=1)
                    .hide(axis="index")
                    .map(color_cell, subset=["Land cover"]))


def severity_df(groups):
    rows = [{"Land cover": classes[int(g["combo"]) // 10],
             "Severity":   SEVERITY[int(g["combo"]) % 10],
             "km²":        g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows).pivot_table(index="Land cover", columns="Severity",
                                        values="km²", aggfunc="sum", fill_value=0)
    df = df[[s for s in SEVERITY.values() if s in df.columns]]
    df = df[df.sum(axis=1) >= 0.05]
    return df.loc[df.sum(axis=1).sort_values().index]


def severity_chart(df):
    colors = [SEV_COLORS[list(SEVERITY.values()).index(c)] for c in df.columns]
    fig, ax = plt.subplots(figsize=(5, 3))
    df.plot(kind="barh", stacked=True, color=colors, ax=ax, width=0.7)
    ax.set_xlabel("Burnt area (km²)")
    ax.set_ylabel("")
    ax.legend(title="Severity", frameon=False, fontsize=8, title_fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(fig)
    return fig


# ---------- Images and indices ----------
def s2_composite(start, end, area, max_cloud=20):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(area)
            .filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
            .median()
            .clip(area))


def nbr(img):
    return img.normalizedDifference(["B8", "B12"]).rename("NBR")


def dnbr_images(geom, before_dates, after_dates):
    before = s2_composite(*before_dates, geom)
    after  = s2_composite(*after_dates, geom)
    dnbr   = nbr(before).subtract(nbr(after)).rename("dNBR")
    return before, after, dnbr


def burnt_mask(dnbr, geom, thr):
    land = WORLDCOVER.clip(geom).neq(80)
    return dnbr.gt(thr).And(land).rename("burnt")


# ---------- Geometry helpers ----------
def edge_km2(mask, geom, scale=100):
    ring = geom.difference(geom.buffer(-500, 10), 10)
    return (ee.Image.pixelArea().updateMask(mask)
            .reduceRegion(reducer=ee.Reducer.sum(), geometry=ring, scale=scale, maxPixels=1e10)
            .get("area"))


def fit_area(geom, before_dates, after_dates, thr, tries=4, step=5000):
    for i in range(tries):
        _, _, dnbr = dnbr_images(geom, before_dates, after_dates)
        touching = ee.Number(edge_km2(burnt_mask(dnbr, geom, thr), geom)).getInfo() or 0
        if touching < 1e5:
            return geom, i
        geom = geom.buffer(step, 10).bounds(10)
    return geom, tries

# ---------- BurnScope science ----------
def burnscope(aoi, before_dates, after_dates, threshold=0.27):
    # 1. Images and dNBR
    before, after, dnbr = dnbr_images(aoi, before_dates, after_dates)

    # 2. Burnt mask (land only) and severity classes
    lc    = WORLDCOVER.clip(aoi)
    burnt = burnt_mask(dnbr, aoi, threshold)
    sev   = (ee.Image(1).where(dnbr.gte(0.27), 2)
                        .where(dnbr.gte(0.44), 3)
                        .where(dnbr.gte(0.66), 4)
                        .updateMask(burnt).rename("severity"))

    # 3. Numbers: total, per land cover, per land cover × severity
    # 3. Numbers: ONE reduction (land cover × severity)
    area  = ee.Image.pixelArea().updateMask(burnt)
    combo = lc.toInt().multiply(10).add(sev).rename("combo")
    sev_groups = (area.addBands(combo)
                  .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="combo"),
                                geometry=aoi, scale=20, maxPixels=1e10)
                  .get("groups")
                  .getInfo())

    # 4. Per-class totals and grand total, computed in Python (instant)
    by_class = {}
    for g in sev_groups:
        c = int(g["combo"]) // 10
        by_class[c] = by_class.get(c, 0) + g["sum"]
    groups = [{"class": c, "sum": s} for c, s in by_class.items()]
    total  = sum(by_class.values())

    return {"before": before, "after": after, "dnbr": dnbr, "burnt": burnt, "severity": sev,
            "km2": total / 1e6,
            "table": lc_table(groups) if groups else None,
            "sev_df": severity_df(sev_groups) if sev_groups else None}

print("Engine ready")

Engine ready


In [40]:
print(f"{len(FIRE_EVENTS)} fire events loaded")

10 fire events loaded


In [41]:
from google.colab import output
output.enable_custom_widget_manager()

In [42]:
import solara, ipyleaflet

# ---------- State ----------
mode         = solara.reactive("Map your own")
event        = solara.reactive(list(FIRE_EVENTS)[-1])
before_start = solara.reactive("2024-07-20")
before_end   = solara.reactive("2024-08-10")
after_start  = solara.reactive("2024-08-15")
after_end    = solara.reactive("2024-09-05")
threshold  = solara.reactive(0.27)
basemap    = solara.reactive("Google satellite")
show_burnt = solara.reactive(True)
area       = solara.reactive(ee.Geometry.Rectangle([23.75, 37.98, 24.10, 38.32]))
result     = solara.reactive(None)
status     = solara.reactive("")

BASEMAPS = {
    "Google satellite": "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    "Google map":       "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    "Google terrain":   "https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}",
}
TRUE_COLOR  = {"bands": ["B4", "B3", "B2"],  "min": 0, "max": 3000}
FALSE_COLOR = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}
SEV_LEGEND  = {"Low (0.10–0.27)":           SEV_COLORS[0],
               "Moderate-low (0.27–0.44)":  SEV_COLORS[1],
               "Moderate-high (0.44–0.66)": SEV_COLORS[2],
               "High (> 0.66)":             SEV_COLORS[3]}
RESULT_LAYERS = ("Before · true color", "After · false color", "Burn severity (dNBR)", "Study area")

# ---------- Map ----------
m = geemap.Map(height="600px")
outline = {"shapeOptions": {"color": "#FFFFFF", "weight": 2, "fillOpacity": 0}}
m.draw_control.rectangle = outline
m.draw_control.polygon = outline
base = m.layers[0]
base.url = BASEMAPS[basemap.value]
base.name = "Basemap"
base.attribution = "Imagery © Google"
m.centerObject(area.value, 11)
m.add_legend(title="Burn severity (dNBR)", legend_dict=SEV_LEGEND, position="bottomright")


# ---------- Map helpers ----------
def set_basemap(name):
    base.url = BASEMAPS[name]


def remove_results():
    for c in list(m.controls):
        if isinstance(c, ipyleaflet.SplitMapControl):
            m.remove_control(c)
    for layer in list(m.layers):
        if layer.name in RESULT_LAYERS:
            m.remove_layer(layer)


def update_map(*args):
    r = result.value
    if r is None:
        return
    remove_results()
    left  = geemap.ee_tile_layer(r["before"], TRUE_COLOR,  "Before · true color")
    right = geemap.ee_tile_layer(r["after"],  FALSE_COLOR, "After · false color")
    m.add_layer(left)
    m.add_layer(right)
    m.add_control(ipyleaflet.SplitMapControl(left_layer=left, right_layer=right))
    m.addLayer(r["severity"], {"min": 1, "max": 4, "palette": SEV_COLORS},
               "Burn severity (dNBR)", show_burnt.value)
    m.addLayer(ee.Image().paint(area.value, 0, 2), {"palette": ["#FFFFFF"]}, "Study area")


# ---------- Actions (what the buttons do) ----------
def analyze():
    status.value = ""
    try:
        result.value = burnscope(area.value,
                                 (before_start.value, before_end.value),
                                 (after_start.value, after_end.value),
                                 threshold.value)
        update_map()
        m.centerObject(area.value)

        touching = (ee.Number(edge_km2(result.value["burnt"], area.value)).getInfo() or 0) / 1e6
        if touching > 0.1:
            status.value = (f"⚠️ The burnt area touches the edge of your box ({touching:.1f} km² along the edge). "
                            "The fire probably continues outside, so draw a bigger box for the full area.")
    except Exception as e:
        status.value = f"⚠️ {e}"


def run():
    try:
        if m.user_roi is not None:
            area.value = m.user_roi
            m.draw_control.clear()
        analyze()
    except Exception as e:
        status.value = f"⚠️ {e}"


def load_event(*args):
    try:
        ev = FIRE_EVENTS[event.value]
        before_start.value, before_end.value = ev["before"]
        after_start.value,  after_end.value  = ev["after"]
        status.value = "🔍 Fitting the box to the fire…"

        geom = ee.Geometry.Rectangle(ev["bbox"]).buffer(3000, 10).bounds(10)
        geom, grown = fit_area(geom, ev["before"], ev["after"], threshold.value)
        area.value = geom
        m.draw_control.clear()
        analyze()

        if grown and not status.value:
            status.value = f"🔍 The box was enlarged {grown} time(s) (+{grown * 5} km) to fit the whole fire."
    except Exception as e:
        status.value = f"⚠️ {e}"


def clear_area(*args):
    remove_results()
    m.draw_control.clear()
    result.value = None


# ---------- Page ----------
@solara.component
def Page():
    with solara.Columns([1, 3]):
        with solara.Column():
            solara.Markdown("# 🔥 BurnScope")
            solara.ToggleButtonsSingle(value=mode, values=["Map your own", "Major events"])

            if mode.value == "Major events":
                solara.Select("Event", value=event, values=list(FIRE_EVENTS))
                ev = FIRE_EVENTS[event.value]
                solara.Markdown(f"📅 Before **{ev['before'][0]} → {ev['before'][1]}** · "
                                f"After **{ev['after'][0]} → {ev['after'][1]}**")
                solara.Markdown(ev["info"])
                solara.Markdown(f"[Source]({ev['source']})")
                solara.SliderFloat("Burnt if dNBR above", value=threshold,
                                   min=0.1, max=0.66, step=0.01, thumb_label="always")
                solara.Button("Map this event", on_click=load_event, color="primary")
            else:
                solara.Markdown("**When · before**")
                solara.InputText("From", value=before_start)
                solara.InputText("To", value=before_end)
                solara.Markdown("**When · after**")
                solara.InputText("From", value=after_start)
                solara.InputText("To", value=after_end)
                solara.SliderFloat("Burnt if dNBR above", value=threshold,
                                   min=0.1, max=0.66, step=0.01, thumb_label="always")
                solara.Button("Map burnt area", on_click=run, color="primary")
                solara.Button("New area", on_click=clear_area, outlined=True)

            solara.Markdown(status.value)
            solara.Markdown("**Map layers**")
            solara.Select("Basemap", value=basemap, values=list(BASEMAPS), on_value=set_basemap)
            solara.Checkbox(label="Show burn severity (dNBR)", value=show_burnt, on_value=update_map)

            r = result.value
            if r is not None:
                solara.Markdown(f"### 🔥 Burnt area: {r['km2']:.1f} km²")
                if r["table"] is not None:
                    solara.display(r["table"])
                if r["sev_df"] is not None:
                    solara.Markdown("**Burnt area by land cover and severity**")
                    solara.FigureMatplotlib(severity_chart(r["sev_df"]))
        with solara.Column():
            solara.display(m)

Page()

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text

In [43]:
m

Map(center=[41.07553826634052, 26.025000000000013], controls=(WidgetControl(options=['position', 'transparent_…